# Loading The Kang Dataset

**REQUIRED DAY 3**

## Already fetched, already cached

Today's dataset — [Kang et al. 2018](https://www.nature.com/articles/nbt.4042), GSE96583 — loads in one line via the scverse ecosystem's `pertpy` package:

```python
import pertpy as pt
adata = pt.dt.kang_2018()
```

That's the real command, shown so you know exactly where this data comes from — but it downloads from a remote host, and twenty people all calling it at once during class is exactly the kind of redundant load Day 2's FASTQ fetch avoided by caching. It's already been run once and saved to `/tscc/nfs/home/juf009/day3_shared_data/kang_2018_checkpoint.h5ad`. Load that instead.

In [ ]:
import scanpy as sc

adata = sc.read_h5ad("/tscc/nfs/home/juf009/day3_shared_data/kang_2018_checkpoint.h5ad")
adata

## What's actually in `.obs`

Real shape: **24,673 cells x 15,706 genes**. Three columns matter today:

- **`label`** — the condition: `stim` (IFN-β, 6h) or `ctrl` (untreated).
- **`replicate`** — the donor (8 lupus patients, e.g. `patient_1015`). **This is the independent unit**, not the cell, not even the sample — each donor contributes one `stim` sample and one `ctrl` sample.
- **`cell_type`** — published annotations (CD4 T cells, CD14+ Monocytes, B cells, NK cells, CD8 T cells, FCGR3A+ Monocytes, Dendritic cells, Megakaryocytes). Day 2 already taught annotation from scratch — today reuses these rather than re-deriving them.

In [ ]:
adata.obs["label"].value_counts()

In [ ]:
adata.obs["replicate"].value_counts()

In [ ]:
adata.obs["cell_type"].value_counts()

## Trust but verify: is `.X` actually raw counts?

Every downstream step today (especially pseudobulk DE in the next notebook) needs raw integer counts, not normalized values. Don't assume — check. `nCount_RNA` is a per-cell total-count column that shipped with the data; if `.X` is raw counts, summing each cell's row should reproduce it exactly.

In [ ]:
import numpy as np

row_sums = np.asarray(adata.X.sum(axis=1)).ravel()
matches = np.allclose(row_sums[:200], adata.obs["nCount_RNA"].values[:200])
print("X row sums match nCount_RNA for first 200 cells:", matches)

If that printed `True`, `.X` is confirmed raw counts — safe to aggregate directly for pseudobulk in the next notebook, no extra layer needed.

## Agent-assisted orientation, done right

> Weak: "Summarize this dataset."
>
> Strong: "This AnnData has a `replicate` column (8 donors) and a `label` column (2 conditions), with each donor appearing in both conditions. Confirm that every donor has exactly one `stim` and one `ctrl` sample, and flag it if any donor is missing a condition or has more than one sample per condition — that would break the paired design the rest of today's analyses depend on."

## Practice

Run the cross-tab below yourself and confirm every donor has exactly one sample per condition — the paired structure every later notebook assumes. This is Agent-B item 11 (can this be reproduced from the provided inputs) in miniature.

In [ ]:
adata.obs.groupby("replicate")["label"].value_counts().unstack()

## Further reading

- [Kang et al. 2018, Nature Biotechnology](https://www.nature.com/articles/nbt.4042)
- [pertpy documentation](https://pertpy.readthedocs.io/)